In [1]:
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, hstack

df = pd.read_csv(
    "cve_preprocessed.csv",
    encoding="utf-8-sig"
)

print(f"원본 데이터 로드: {len(df)}행")
print("데이터 크기:", df.shape)

display(df.head())

원본 데이터 로드: 9994행
데이터 크기: (9994, 9)


,cve_id,cvss_score,severity,attack_vector,attack_complexity,privileges_required,user_interaction,cwe,description
0,CVE-2026-57752,8.5,HIGH,NETWORK,LOW,LOW,NONE,CWE-89,Contributor SQL Injection in iNET Webkit 1.2.4...
1,CVE-2026-57753,5.3,MEDIUM,NETWORK,LOW,NONE,NONE,CWE-497,Unauthenticated Sensitive Data Exposure in Kit...
2,CVE-2026-57754,6.5,MEDIUM,NETWORK,LOW,LOW,REQUIRED,CWE-79,Contributor Cross Site Scripting (XSS) in Live...
3,CVE-2026-57755,6.5,MEDIUM,NETWORK,LOW,LOW,REQUIRED,CWE-79,Contributor Cross Site Scripting (XSS) in Mosa...
4,CVE-2026-57756,8.5,HIGH,NETWORK,LOW,LOW,NONE,CWE-89,Contributor SQL Injection in nicen-localize-im...


In [2]:
required_columns = [
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cwe",
    "description",
    "severity",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"필수 컬럼이 없습니다: {missing_columns}"
    )

# CWE 및 Description 결측치 처리
df["cwe"] = (
    df["cwe"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.upper()
)

df["description"] = (
    df["description"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("결측치 확인")
print(df[required_columns].isnull().sum())

결측치 확인
attack_vector          0
attack_complexity      0
privileges_required    0
user_interaction       0
cwe                    0
description            0
severity               0
dtype: int64


In [3]:
feature_cols = [
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cwe",
]

# 범주형 Feature를 One-Hot Encoding
encoded_features = pd.get_dummies(
    df[feature_cols],
    prefix=feature_cols,
    dtype=int,
)

print("One-Hot Encoding 완료")
print("범주형 Feature 크기:", encoded_features.shape)

display(encoded_features.head())

One-Hot Encoding 완료
범주형 Feature 크기: (9994, 368)


,attack_vector_ADJACENT_NETWORK,attack_vector_LOCAL,attack_vector_NETWORK,attack_vector_PHYSICAL,attack_complexity_HIGH,attack_complexity_LOW,privileges_required_HIGH,privileges_required_LOW,privileges_required_NONE,user_interaction_NONE,...,cwe_CWE-93,cwe_CWE-939,cwe_CWE-94,cwe_CWE-940,cwe_CWE-942,cwe_CWE-943,cwe_CWE-95,cwe_CWE-98,cwe_CWE-99,cwe_UNKNOWN
0,0,0,1,0,0,1,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,1,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,1,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,1,0,0,1,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0


In [4]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=300,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
)

description_features = tfidf_vectorizer.fit_transform(
    df["description"]
)

print("TF-IDF 변환 완료")
print(
    "Description Feature 크기:",
    description_features.shape
)

print(
    "TF-IDF 단어 예시:",
    tfidf_vectorizer.get_feature_names_out()[:20]
)

TF-IDF 변환 완료
Description Feature 크기: (9994, 300)
TF-IDF 단어 예시: ['10' '11' '12' '13' '14' '149' '149 7827' '15' '150' '150 7871' '17' '18'
 '19' '2026' '24' '26' '47' '47 allowed' '53' '53 allowed']


In [5]:
label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    df["severity"]
)

print("Label Encoding 완료")

label_mapping = {
    class_name: int(encoded_value)
    for encoded_value, class_name
    in enumerate(label_encoder.classes_)
}

print("Label 매핑:", label_mapping)

Label Encoding 완료
Label 매핑: {'CRITICAL': 0, 'HIGH': 1, 'LOW': 2, 'MEDIUM': 3}


In [6]:
# One-Hot Encoding 결과를 희소 행렬로 변환
categorical_features = csr_matrix(
    encoded_features.values
)

# 범주형 Feature + Description TF-IDF 결합
X = hstack(
    [
        categorical_features,
        description_features,
    ],
    format="csr",
)

print("최종 Feature 결합 완료")
print("X Shape:", X.shape)
print("y Shape:", y.shape)

최종 Feature 결합 완료
X Shape: (9994, 668)
y Shape: (9994,)


In [7]:
categorical_feature_names = (
    encoded_features.columns.tolist()
)

description_feature_names = [
    f"description_{word}"
    for word in tfidf_vectorizer.get_feature_names_out()
]

all_feature_names = (
    categorical_feature_names
    + description_feature_names
)

print("전체 Feature 개수:", len(all_feature_names))
print("Feature 예시:", all_feature_names[:20])

전체 Feature 개수: 668
Feature 예시: ['attack_vector_ADJACENT_NETWORK', 'attack_vector_LOCAL', 'attack_vector_NETWORK', 'attack_vector_PHYSICAL', 'attack_complexity_HIGH', 'attack_complexity_LOW', 'privileges_required_HIGH', 'privileges_required_LOW', 'privileges_required_NONE', 'user_interaction_NONE', 'user_interaction_REQUIRED', 'cwe_CWE-1021', 'cwe_CWE-1023', 'cwe_CWE-1025', 'cwe_CWE-1027', 'cwe_CWE-1058', 'cwe_CWE-1077', 'cwe_CWE-1104', 'cwe_CWE-1124', 'cwe_CWE-113']


In [8]:
feature_data = {
    "X": X,
    "y": y,
    "label_encoder": label_encoder,
    "tfidf_vectorizer": tfidf_vectorizer,
    "categorical_columns": feature_cols,
    "encoded_columns": categorical_feature_names,
    "feature_names": all_feature_names,
}

joblib.dump(
    feature_data,
    "feature_data.pkl"
)

print("feature_data.pkl 저장 완료")

feature_data.pkl 저장 완료
